<a href="https://colab.research.google.com/github/apar-tech/rag-chatbot/blob/main/phase2/query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    !pip install -q \
    chromadb \
    google-genai \
    groq

    print("Libraries installed successfully!")

except Exception as e:
    print("Installation Error:")
    print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [3]:
# Check File Sizes
import os

try:
    print("chunks.pkl size:", os.path.getsize("chunks.pkl"))
    print("chunk_embeddings.pkl size:", os.path.getsize("chunk_embeddings.pkl"))
except FileNotFoundError as e:
    print("File not found error:")
    print(e)
except Exception as e:
    print("Unexpected error:")
    print(e)

chunks.pkl size: 198693
chunk_embeddings.pkl size: 2097152


In [4]:
#  Load Data and Create ChromaDB Collection
try:
    import pickle
    import chromadb

    # Load saved data
    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("chunk_embeddings.pkl", "rb") as f:
        chunk_embeddings = pickle.load(f)

    print("Files loaded successfully!")

    # Create ChromaDB client
    chroma_client = chromadb.PersistentClient(
        path="./digital_chromadb"
    )

    # Create collection
    collection = chroma_client.get_or_create_collection(
        name="digital_docs"
    )

    # Add data to collection
    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_embeddings
    )

    print("Collection created successfully!")
    print("Records:", collection.count())

except FileNotFoundError as e:
    print("File not found error:")
    print(e)
except pickle.UnpicklingError as e:
    print("Pickle loading error:")
    print(e)
except Exception as e:
    print("Error:")
    print(e)

Files loaded successfully!
Collection created successfully!
Records: 111


In [5]:
#  Initialize Gemini
try:
    from google import genai
    from google.colab import userdata

    client = genai.Client(
        api_key=userdata.get("geminikey")
    )

    print("Gemini initialized successfully!")

except ImportError as e:
    print("Import error:")
    print(e)
except Exception as e:
    print("Gemini Error:")
    print(e)

Gemini initialized successfully!


In [6]:
#  Generate Query Embedding
try:
    query = "What is Artificial intelligence?"

    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=query
    )

    query_embedding = response.embeddings[0].values

    print("Query embedding generated!")

except AttributeError as e:
    print("Attribute error - check response structure:")
    print(e)
except Exception as e:
    print("Embedding Error:")
    print(e)

Query embedding generated!


In [7]:
# Retrieve Similar Chunks
try:
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    print("Retrieved top 5 chunks!")

    retrieved_chunks = results["documents"][0]

    for i, chunk in enumerate(retrieved_chunks):
        print("=" * 60)
        print(f"Chunk {i + 1}")
        print(chunk[:500])
        print()

except KeyError as e:
    print("Key error - check results structure:")
    print(e)
except Exception as e:
    print("Error:", e)

Retrieved top 5 chunks!
Chunk 1
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-pr

Chunk 2
Russell and Norvig agree with Turing that intelligence must be defined in terms of external behavior, not internal structure. However, they are critical that the test requires the machine to imitate humans. "Aeronautical engineering texts", they wrote, "do not define the goal of their field as making 'machines that fly so exactly like pigeons that they can fool other pigeons.'" AI founder John McCarthy agreed, writing that "Artificial intelligence is not

In [8]:
# Initialize Groq
try:
    from groq import Groq
    from google.colab import userdata

    groq_client = Groq(
        api_key=userdata.get("groqkey")
    )

    print("Groq initialized successfully!")
    context = "\n\n".join(retrieved_chunks)

    print("Context created successfully!")

except ImportError as e:
    print("Import error:")
    print(e)
except Exception as e:
    print("Groq Error:")
    print(e)

Groq initialized successfully!
Context created successfully!


In [9]:
#  Generate Answer
try:
    prompt = f"""
You are a networking assistant.

Answer only using the provided context.

If the answer is not found, reply:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{query}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    print(answer)

except AttributeError as e:
    print("Attribute error - check response structure:")
    print(e)
except Exception as e:
    print("Generation Error:")
    print(e)

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.


In [14]:
#  Define Reusable Function
try:
    def ask_document_question(question):
        try:
            # Generate embedding for the question
            embedding_response = client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=question
            )

            query_embedding = embedding_response.embeddings[0].values

            # Retrieve relevant chunks
            search_results = collection.query(
                query_embeddings=[query_embedding],
                n_results=5
            )

            context = "\n\n".join(
                search_results["documents"][0]
            )

            # Generate answer using Groq
            prompt = f"""
You are a networking assistant.

Answer only using the provided context.

Context:
{context}
Question:
{question}
"""

            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            return response.choices[0].message.content

        except KeyError as e:
            return f"Key error in search results: {e}"
        except AttributeError as e:
            return f"Attribute error: {e}"
        except Exception as e:
            return f"Error in processing question: {e}"

    print("Function defined successfully!")

except Exception as e:
    print("Function definition error:")
    print(e)

Function defined successfully!


In [15]:
#  Test Function - Question 1
try:
    result = ask_document_question(
        "What is the purpose of Machine learning?"
    )
    print(result)

except NameError as e:
    print("Function not defined:")
    print(e)
except Exception as e:
    print("Error calling function:")
    print(e)

The purpose of machine learning is to develop and study statistical algorithms that can learn from data and generalize to unseen data, allowing them to perform tasks without being explicitly programmed.


In [16]:
#  Test Function - Question 2
try:
    result = ask_document_question(
        "Explain machine learning, deep learning and Artificial intelligence?"
    )
    print(result)

except NameError as e:
    print("Function not defined:")
    print(e)
except Exception as e:
    print("Error calling function:")
    print(e)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. 

Deep learning (DL) is a type of machine learning that focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. It takes inspiration from biological neuroscience and revolves around stacking artificial neurons into layers and "training" them to process data.

Artificial intelligence (AI) encompasses machine learning, which is the study of programs that can improve their performance on a given task automatically. AI has been a part of machine learning from the beginning, and it allows programs to read, write, and communicate in human languages through natural language processing (NLP).
